# Composable CDP · Executive Story & One Customer End-to-End

This notebook is the **15-minute guided tour** of the **Composable Customer Data Platform (CDP)** built 100% natively in **BigQuery**.

It combines two complementary breakthroughs inside a single SQL-native warehouse pipeline:
1. **Bipartite Identifier Graph Rarity & 2-Hop Neighbourhood Features (`40_block.sql` + `50_candidates.sql`)** — replacing fragile external PyTorch/GraphSAGE training pipelines with closed-form Inverse Document Frequency (`LN(1 + N / degree)`) and explicit 2-hop neighbourhood comparisons (`unshared_email_both`, `unshared_phone_both`, `street_conflict`).
2. **Hybrid Vector + Lexical Search (`AI.SEARCH` / `VECTOR_SEARCH` + Reciprocal Rank Fusion) + Gemini 3.5 Flash Adjudication (`AI.GENERATE`) + Pass-2 Contradiction Pruning (`60_adjudicate.sql` + `70_graph.sql`)** — catching fuzzy semantic matches (transliterations, nicknames, unstructured call transcripts) while keeping false merges to **4 across the planted hard cases** (`99.16%` corpus-wide pairwise precision).

### The 5-Act Notebook Suite
| Notebook | Act | Focus |
|---|---|---|
| **`01_identity_resolution_story.ipynb`** *(this notebook)* | **Overview** | Executive 3-method benchmark + following **one customer (`Abby Noland`, `PER-1cdfdb9f52ddb25a`)** end-to-end |
| **`02_graph_pathology_and_rarity.ipynb`** | **Act 1** | Why naive graph connected components collapses 5,195 profiles into a hairball, and how IDF Rarity fixes it |
| **`03_hybrid_search_and_2hop_features.ipynb`** | **Act 2** | Why Hybrid Search (`AI.SEARCH` + RRF) + 2-Hop SQL Graph Features beat standalone GraphSAGE GNNs |
| **`04_llm_adjudicator_and_contradiction_guard.ipynb`** | **Act 3** | Gemini 3.5 Flash grey-zone adjudication, prompt-injection defense, and Pass-2 transitive contradiction pruning |
| **`05_semantic_graph_governance_and_roi.ipynb`** | **Act 4 & 5** | Field survivorship, intersection consent (Privacy Act 1988), `GRAPH_EXPAND` semantic analytics, and measured cost |

In [1]:
%load_ext bigquery_magics

import bigquery_magics
import pandas as pd
from IPython.display import display
from pathlib import Path

# ---------------------------------------------------------------------------
# Set PROJECT if running outside a checkout that has demo/config.env
PROJECT = ""          # e.g. "all-things-cdp"
DATASET = ""          # leave blank for the default, "cdp"
# ---------------------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 52)

CFG, CONFIG, _source = {}, None, None

def _find_config():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for candidate in (base / "config.env", base / "demo" / "config.env"):
            if candidate.is_file():
                return candidate
    return None

if not PROJECT:
    CONFIG = _find_config()
    if CONFIG:
        for _line in CONFIG.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                CFG[_k.strip()] = _v.split("#", 1)[0].strip().strip('"').strip("'")
        PROJECT = CFG.get("CDP_PROJECT", "")
        _source = str(CONFIG)

if not PROJECT:
    import os
    PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT") or "all-things-cdp"
    _source = "environment / default"

DATASET = DATASET or CFG.get("CDP_DS") or CFG.get("CDP_DATASET_PREFIX") or "cdp"
LOCATION = CFG.get("CDP_LOCATION", "US")

bigquery_magics.context.project = PROJECT
bigquery_magics.context.location = LOCATION
bigquery_magics.context.progress_bar_type = None

print(f"project    {PROJECT}   (from {_source})")
print(f"dataset    {DATASET}   (location {LOCATION})")

project    all-things-cdp   (from demo/config.env)
dataset    cdp   (location US)

## 1 · The Headline Benchmark: 3 Resolution Methods on the Same 20,000 Profiles

Before following a single customer, let's inspect the corpus-wide benchmark from `cdp.v_scorecard` and `cdp.resolution_runs`. Every method runs over the exact same **20,000 source records** (`8,000` true people across 7 source systems: `CRM`, `LOYALTY`, `ECOM`, `SUPPORT`, `CALL`, `POS`, `ENRICH`):

1. **`baseline_cc` (Naive Connected Components):** Any shared identifier links two profiles. Seven store-kiosk and contact-centre hub identifiers (`degree = 419..575`) collapse **5,195 distinct customers into a single giant hairball**, driving precision down to **`0.06%`** (`13,488,214` false-positive pairs).
2. **`weighted_cc` (IDF-Weighted Graph Projection Only):** Drops promiscuous hubs (`degree > 25`) and requires cumulative shared-identifier rarity `weight >= 7.0`. Eliminates the hairball (`95.93%` precision, `321` FPs), but over-merges households sharing a landline/email (`100` FPs) and stale recycled identifiers (`38` FPs), while missing fuzzy semantic matches (`40.30%` recall).
3. **`composable_cdp` (2-Hop Graph + Hybrid Search + Gemini 3.5 Flash + Pass-2 Contradiction Guard):** Combines 2-hop graph rarity with hybrid lexical/vector retrieval, Gemini 3.5 Flash grey-zone adjudication, and Pass-2 transitive contradiction pruning. With every grey-zone pair judged, it reaches **`99.16%` pairwise precision (`74` FPs total vs `321` in `weighted_cc`)**, **`46.77%` recall (`8,782` TPs vs `7,566` in `weighted_cc`)**, **`0.6356` F1**, and **zero hairballs**. Recall is the honest weak spot: more than half of true pairs are still missed, and that is a deliberate bias toward under-merging (a missed match costs a duplicate mailing; a false merge is a privacy incident).

In [2]:
%%bigquery df_runs
SELECT
  r.method,
  r.threshold,
  r.n_profiles,
  r.n_components,
  r.largest_component,
  r.n_hairballs,
  r.profiles_in_hairballs
FROM `cdp.resolution_runs` AS r
ORDER BY CASE r.method
  WHEN 'baseline_cc'    THEN 1
  WHEN 'weighted_cc'    THEN 2
  WHEN 'composable_cdp' THEN 3
END

method,threshold,n_profiles,n_components,largest_component,n_hairballs,profiles_in_hairballs
baseline_cc,0.0,20000,11049,5195,1,5195
weighted_cc,7.0,20000,14091,5,0,0
composable_cdp,0.92,20000,13566,9,0,0


In [3]:
%%bigquery df_method_comp
SELECT
  pathology,
  method,
  tp,
  fp,
  fn,
  ROUND(precision, 4) AS precision,
  ROUND(recall, 4)    AS recall,
  ROUND(f1, 4)        AS f1
FROM `cdp.v_method_comparison`
WHERE pathology IN ('ALL', 'hub', 'household', 'supersession', 'semantic_fuzzy')
ORDER BY
  CASE pathology
    WHEN 'ALL' THEN 0 WHEN 'hub' THEN 1 WHEN 'household' THEN 2
    WHEN 'supersession' THEN 3 WHEN 'semantic_fuzzy' THEN 4
  END,
  CASE method WHEN 'baseline_cc' THEN 1 WHEN 'weighted_cc' THEN 2 ELSE 3 END

pathology,method,tp,fp,fn,precision,recall,f1
ALL,baseline_cc,8064,13488214,10712,0.0006,0.4295,0.0012
ALL,weighted_cc,7566,321,11210,0.9593,0.403,0.5675
ALL,composable_cdp,8782,74,9994,0.9916,0.4677,0.6356
hub,baseline_cc,2557,13488039,2883,0.0002,0.47,0.0004
hub,weighted_cc,2077,64,3363,0.9701,0.3818,0.5479
hub,composable_cdp,2404,19,3036,0.9922,0.4419,0.6115
household,baseline_cc,151,54,87,0.7366,0.6345,0.6817
household,weighted_cc,147,100,91,0.5951,0.6176,0.6062
household,composable_cdp,152,2,86,0.987,0.6387,0.7755
supersession,baseline_cc,28,29,62,0.4912,0.3111,0.381


## 2 · Pick One Customer and Follow Them End-to-End

Now let's follow **one resolved person** through every stage of the pipeline:
1. Pick a customer with records across **5 different source systems**, including an **unstructured support ticket**, where the **Gemini 3.5 Flash adjudicator** had to rule on grey-zone pairs.
2. Inspect their **raw, uncleaned records** (`party_records`).
3. See what **Stage 20 Normalisation** fixes (`0489 909 604` $\rightarrow$ `+61489909604`) and what it cannot fix (`abyb noland`, truncated postcode `42`, transposed postcode `4202`).
4. Trace how **Hybrid Search + 2-Hop Graph Rarity** scores every candidate pair (`pair_tiers`).
5. Inspect the **Golden Person** (`golden_person`) and **Field Survivorship Provenance** (`field_survivorship`).

In [4]:
%%bigquery df_subject
WITH members AS (
  SELECT person_id, record_id FROM `cdp.person_assignment`
),
stats AS (
  SELECT m.person_id,
         COUNT(*)                                        AS n_records,
         COUNT(DISTINCT pr.source_system)                AS n_sources,
         STRING_AGG(DISTINCT pr.source_system ORDER BY pr.source_system) AS sources,
         COUNTIF(pr.source_system IN ('CALL','SUPPORT')) AS n_unstructured
  FROM members AS m
  JOIN `cdp.party_records` AS pr USING (record_id)
  GROUP BY m.person_id
),
grey AS (
  SELECT a.person_id, COUNT(*) AS n_grey
  FROM `cdp.pair_tiers` AS t
  JOIN members AS a ON a.record_id = t.record_id_a
  JOIN members AS b ON b.record_id = t.record_id_b AND b.person_id = a.person_id
  WHERE t.tier = 'GREY_ZONE'
  GROUP BY a.person_id
)
SELECT s.person_id,
       gp.full_name, gp.city, gp.postcode,
       s.n_records, s.n_sources, s.sources, s.n_unstructured,
       IFNULL(g.n_grey, 0) AS grey_zone_pairs
FROM stats AS s
LEFT JOIN grey AS g USING (person_id)
LEFT JOIN `cdp.golden_person` AS gp USING (person_id)
-- Pinned to the person who owns CRM record CRM-5d268dd4 (Abby Noland) so the
-- narrative below stays stable across re-runs. She was originally picked by
-- the criteria kept here: >= 3 sources, >= 1 unstructured record, 4-8 records.
WHERE s.person_id = (SELECT person_id FROM members WHERE record_id = 'CRM-5d268dd4')
  AND s.n_sources >= 3 AND s.n_unstructured >= 1 AND s.n_records BETWEEN 4 AND 8

person_id,full_name,city,postcode,n_records,n_sources,sources,n_unstructured,grey_zone_pairs
PER-1cdfdb9f52ddb25a,Abby Noland,Gold Coast,4220,5,5,"CRM,ENRICH,LOYALTY,POS,SUPPORT",1,2


In [5]:
SUBJECT   = df_subject.iloc[0]
PERSON_ID = str(SUBJECT["person_id"])
params    = {"pid": PERSON_ID}
print(f"Following {SUBJECT['full_name']} ({PERSON_ID}) across {SUBJECT['n_records']} records in {SUBJECT['sources']}")

Following Abby Noland (PER-1cdfdb9f52ddb25a) across 5 records in CRM,ENRICH,LOYALTY,POS,SUPPORT


## 3 · Raw vs Normalised Records Across 5 Source Systems

Look closely at the 5 records for **Abby Noland** below:
- **`CRM-5d268dd4` (`CRM`, trust 9):** Complete record (`Abby Noland`, `anoland@yahoo.com.au`, `+61489909604`, postcode `4220`, DOB `1955-07-27`).
- **`LOY-ac677db5` (`LOYALTY`, trust 7):** Misspelled given name (`abyb noland`), local phone format (`0489 909 604` $ightarrow$ normalised to `+61489909604`), truncated postcode (`42` $ightarrow$ dropped as invalid), and loyalty account `ACC-8504064`.
- **`SUP-da73f0e1` (`SUPPORT`, trust 4):** Unstructured support ticket where `AI.GENERATE` extracted `Abby Noland` and `anoland@yahoo.com.au`, but no phone, postcode, or DOB.
- **`POS-1b222aed` (`POS`, trust 3):** Point-of-sale stub with initial-only name (`A Noland`), truncated postcode (`42`), and loyalty card `ACC-8504064`.
- **`ENR-31fd9a08` (`ENRICH`, trust 2):** Third-party enrichment record with `Abby Noland`, `anoland@yahoo.com.au`, and transposed postcode `4202` (instead of `4220`).

In [6]:
%%bigquery df_records --params $params
SELECT pr.record_id, pr.source_system, pr.source_trust,
       pr.raw_name, pr.name_norm,
       pr.raw_email, pr.email_norm,
       pr.raw_phone, pr.phone_e164,
       pr.raw_postcode, pr.postcode_norm,
       pr.dob, pr.account_number, pr.identity_strength
FROM `cdp.person_assignment` AS pa
JOIN `cdp.party_records`     AS pr USING (record_id)
WHERE pa.person_id = @pid
ORDER BY pr.source_trust DESC, pr.record_id

record_id,source_system,source_trust,raw_name,name_norm,raw_email,email_norm,raw_phone,phone_e164,raw_postcode,postcode_norm,dob,account_number,identity_strength
CRM-5d268dd4,CRM,9,Abby Noland,ABBY NOLAND,anoland@yahoo.com.au,anoland@yahoo.com.au,+61489909604,+61489909604,4220,4220,1955-07-27,,4
LOY-ac677db5,LOYALTY,7,abyb noland,ABYB NOLAND,,,0489 909 604,+61489909604,42,,1955-07-27,ACC-8504064,3
SUP-da73f0e1,SUPPORT,4,Abby Noland,ABBY NOLAND,anoland@yahoo.com.au,anoland@yahoo.com.au,,,,,,,1
POS-1b222aed,POS,3,A Noland,A NOLAND,,,,,42,,,ACC-8504064,1
ENR-31fd9a08,ENRICH,2,Abby Noland,ABBY NOLAND,anoland@yahoo.com.au,anoland@yahoo.com.au,,,4202,4202,,,2


## 4 · Pairwise Ranking: Hybrid Retrieval + 2-Hop Graph Rarity + Tiering

Below are the candidate pairs evaluated inside Abby Noland's cluster (`cdp.pair_tiers`):
- Notice `idf_weight_sum` and `min_shared_degree` from the **Bipartite Identifier Graph (`40_block.sql`)**:
  - `CRM-5d268dd4 ~ LOY-ac677db5` shares a unique phone (`degree = 2`, `idf_weight = 9.21`) and exact DOB (`1955-07-27`). Even though `LOY-ac677db5` spelled her name `abyb noland`, the pipeline auto-matches them (`combined_score = 1.000`).
  - `LOY-ac677db5 ~ POS-1b222aed` shares loyalty account `ACC-8504064` (`degree = 2`, `idf_weight = 9.21`) with compatible forename (`ABYB` vs `A`), linking the sparse POS stub (`AUTO_MATCH`).
  - `CRM-5d268dd4 ~ SUP-da73f0e1` and `ENR-31fd9a08 ~ SUP-da73f0e1` link the unstructured support ticket via shared email (`degree = 3`, `idf_weight = 8.80`), routing through the **`GREY_ZONE`** where **Gemini 3.5 Flash** confirms the match.

In [7]:
%%bigquery df_pairs --params $params
SELECT t.record_id_a, t.record_id_b, t.retrieved_by, t.strong_signals,
       ROUND(t.rule_score, 3)      AS rule_score,
       ROUND(t.similarity, 3)      AS similarity,
       ROUND(t.combined_score, 3)  AS combined_score,
       ROUND(t.idf_weight_sum, 2)  AS idf_weight_sum,
       t.min_shared_degree,
       t.unshared_email_both,
       t.tier, t.tier_reason
FROM `cdp.pair_tiers` AS t
JOIN `cdp.person_assignment` AS ma ON ma.record_id = t.record_id_a
JOIN `cdp.person_assignment` AS mb ON mb.record_id = t.record_id_b
WHERE ma.person_id = @pid AND mb.person_id = @pid
ORDER BY t.combined_score DESC

record_id_a,record_id_b,retrieved_by,strong_signals,rule_score,similarity,combined_score,idf_weight_sum,min_shared_degree,unshared_email_both,tier,tier_reason
CRM-5d268dd4,LOY-ac677db5,BOTH,2,1.0,0.905,1.0,9.21,2,False,AUTO_MATCH,Two independent strong identifiers agree with no contradicting evidence.
CRM-5d268dd4,ENR-31fd9a08,BOTH,1,0.97,0.904,0.97,8.8,3,False,AUTO_MATCH,Score above the auto-match threshold with no contradicting evidence.
ENR-31fd9a08,SUP-da73f0e1,BOTH,1,0.85,0.866,0.855,8.8,3,False,GREY_ZONE,"One strong identifier agrees, but not enough to act on alone."
CRM-5d268dd4,SUP-da73f0e1,BOTH,1,0.85,0.826,0.85,8.8,3,False,GREY_ZONE,"One strong identifier agrees, but not enough to act on alone."
LOY-ac677db5,POS-1b222aed,BOTH,1,0.795,0.79,0.795,9.21,2,False,AUTO_MATCH,"Shared loyalty account number, forenames consistent, nothing contradicting."
ENR-31fd9a08,LOY-ac677db5,BOTH,0,0.384,0.789,0.525,0.0,,False,REJECT,Below the rejection threshold.
POS-1b222aed,SUP-da73f0e1,SEMANTIC_ONLY,0,0.295,0.739,0.451,0.0,,False,REJECT,POS stub has an unshared loyalty account or no postcode district and zero strong signals.
LOY-ac677db5,SUP-da73f0e1,SEMANTIC_ONLY,0,0.264,0.722,0.424,0.0,,False,REJECT,Below the rejection threshold.
CRM-5d268dd4,POS-1b222aed,LEXICAL_ONLY,0,0.295,,0.295,0.0,,False,REJECT,POS stub has an unshared loyalty account or no postcode district and zero strong signals.
ENR-31fd9a08,POS-1b222aed,LEXICAL_ONLY,0,0.295,,0.295,0.0,,False,REJECT,POS stub has an unshared loyalty account or no postcode district and zero strong signals.


## 5 · Survivorship & The Golden Record

Once the 5 records are resolved into `PER-1cdfdb9f52ddb25a`, Stage 80 (`80_survivorship.sql`) selects the surviving value for each attribute weighted by `source_trust` and recency, recording full field-level provenance in `cdp.field_survivorship`.

In [8]:
%%bigquery df_survivorship --params $params
SELECT field, surviving_value, won_from_source, source_trust,
       asserting_sources, distinct_values, was_contested, provenance
FROM `cdp.field_survivorship`
WHERE person_id = @pid
ORDER BY was_contested DESC, field

field,surviving_value,won_from_source,source_trust,asserting_sources,distinct_values,was_contested,provenance
address,"33 Fraser Parade, Burleigh Heads QLD",CRM,9,1,3,True,Selected from CRM (trust 9) over 2 competing value(s); corroborated by 1 source(s).
forename,Abby,CRM,9,3,3,True,Selected from CRM (trust 9) over 2 competing value(s); corroborated by 3 source(s).
full_name,Abby Noland,CRM,9,3,3,True,Selected from CRM (trust 9) over 2 competing value(s); corroborated by 3 source(s).
postcode,4220,CRM,9,1,2,True,Selected from CRM (trust 9) over 1 competing value(s); corroborated by 1 source(s).
account_number,ACC-8504064,LOYALTY,7,2,1,False,Selected from LOYALTY (trust 7) — uncontested; corroborated by 2 source(s).
city,Gold Coast,CRM,9,2,1,False,Selected from CRM (trust 9) — uncontested; corroborated by 2 source(s).
dob,1955-07-27,CRM,9,2,1,False,Selected from CRM (trust 9) — uncontested; corroborated by 2 source(s).
email,anoland@yahoo.com.au,CRM,9,3,1,False,Selected from CRM (trust 9) — uncontested; corroborated by 3 source(s).
phone,+61489909604,CRM,9,2,1,False,Selected from CRM (trust 9) — uncontested; corroborated by 2 source(s).
surname,Noland,CRM,9,5,1,False,Selected from CRM (trust 9) — uncontested; corroborated by 5 source(s).


## 6 · Full Run Scorecard (`cdp.v_scorecard`)

Finally, here is the complete single-row run scorecard (`cdp.v_scorecard`) audited against hidden ground truth (`cdp_truth.truth_membership`).

In [9]:
%%bigquery df_scorecard
SELECT
  records,
  true_people,
  predicted_people,
  clusters_containing_multiple_people,
  people_split_across_clusters,
  pairwise_precision,
  pairwise_recall,
  pairwise_f1,
  tp,
  fp,
  fn,
  pairs_evaluated,
  pairs_sent_to_llm,
  pct_of_pairs_using_an_llm,
  adjudicator_model,
  prompt_version
FROM `cdp.v_scorecard`

records,true_people,predicted_people,clusters_containing_multiple_people,people_split_across_clusters,pairwise_precision,pairwise_recall,pairwise_f1,tp,fp,fn,pairs_evaluated,pairs_sent_to_llm,pct_of_pairs_using_an_llm,adjudicator_model,prompt_version
20000,8000,13566,20,4282,0.9916,0.4677,0.6356,8782,74,9994,10773842,41769,0.388,gemini-3.5-flash,adjudicator-v2
